In [1]:
# ============================================================
# 문진톡톡 - 서울아산병원 질환백과 호흡기질환 diseases.json 생성기
# ============================================================

import sys
import subprocess
import importlib.util
from pathlib import Path

# ------------------------------------------------------------
# 0. 패키지 확인/설치
# ------------------------------------------------------------
packages = [
    ("requests", "requests"),
    ("beautifulsoup4", "bs4"),
    ("pandas", "pandas"),
]

for pip_name, import_name in packages:
    if importlib.util.find_spec(import_name) is None:
        print(f"[INSTALL] {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
    else:
        print(f"[OK] {pip_name}")

# ------------------------------------------------------------
# 1. 기본 설정
# ------------------------------------------------------------
import re
import json
import time
import hashlib
import datetime as dt
from typing import Any, Dict, Iterable, List, Optional, Tuple
from urllib.parse import urljoin, urlparse, urlunparse, parse_qsl, urlencode

import requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.amc.seoul.kr"
LIST_URL = "https://www.amc.seoul.kr/asan/healthinfo/disease/diseaseList.do?diseaseKindId=C000019"
DISEASE_KIND_ID = "C000019"
DISEASE_KIND_NAME = "호흡기질환"

OUT_DIR = Path("output_amc_diseases_fixed")
CACHE_DIR = OUT_DIR / "cache_html"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 너무 자주 요청하지 않기 위한 설정
REQUEST_DELAY_SEC = 1.5
USE_CACHE = True
FORCE_REFRESH = False
DETAIL_LIMIT = None       # 테스트만 하려면 5 등으로 변경. 전체는 None.
MAX_LIST_PAGES = 10

print("설정 완료")
print("OUT_DIR:", OUT_DIR.resolve())
print("CACHE_DIR:", CACHE_DIR.resolve())

# ------------------------------------------------------------
# 2. 공통 유틸
# ------------------------------------------------------------
def now_utc_iso() -> str:
    return (
        dt.datetime.now(dt.timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )


def normalize_text(text: Any) -> str:
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def dedupe_keep_order(values: Iterable[str]) -> List[str]:
    seen = set()
    result = []
    for value in values:
        value = normalize_text(value)
        if not value:
            continue
        if value in seen:
            continue
        seen.add(value)
        result.append(value)
    return result


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def hash_url(url: str) -> str:
    return hashlib.sha1(url.encode("utf-8")).hexdigest()


def cache_path_for_url(url: str) -> Path:
    return CACHE_DIR / f"{hash_url(url)}.html"


def set_query_params(url: str, **params: Any) -> str:
    parsed = urlparse(url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    for key, value in params.items():
        if value is None:
            query.pop(key, None)
        else:
            query[key] = str(value)
    return urlunparse(
        (
            parsed.scheme,
            parsed.netloc,
            parsed.path,
            parsed.params,
            urlencode(query),
            parsed.fragment,
        )
    )


def parse_content_id(url: str) -> Optional[str]:
    match = re.search(r"contentId=(\d+)", url)
    return match.group(1) if match else None


def make_disease_id(content_id: str) -> str:
    return f"amc:{content_id}"


def split_final_parenthetical(text: str) -> Tuple[str, Optional[str]]:
    """
    마지막 괄호쌍을 찾되, 괄호 내부에 다시 괄호가 있는 경우도 처리한다.
    """
    text = normalize_text(text)
    if not text.endswith(")"):
        return text, None

    end = len(text) - 1
    depth = 0
    start = None

    for i in range(end, -1, -1):
        ch = text[i]
        if ch == ")":
            depth += 1
        elif ch == "(":
            depth -= 1
            if depth == 0:
                start = i
                break

    if start is None:
        return text, None

    before = normalize_text(text[:start])
    inside = normalize_text(text[start + 1:end])
    return before, inside or None


def split_disease_name(display_title: str) -> Tuple[str, Optional[str]]:
    ko, en = split_final_parenthetical(display_title)
    if en and re.search(r"[A-Za-z]", en):
        return ko, en
    return normalize_text(display_title), None


def is_disease_title_text(text: str) -> bool:
    """
    질환 카드 제목인지 판별한다.
    기존 버그: r"\(([^()]*)\)$"는 영문명 안에 괄호가 있으면 실패했다.
    수정: 마지막 바깥 괄호를 직접 찾고, 그 내부에 영문자가 있으면 질환 제목으로 인정한다.
    """
    text = normalize_text(text)
    if not text:
        return False
    ko, en = split_final_parenthetical(text)
    if not en:
        return False
    if not re.search(r"[A-Za-z]", en):
        return False
    if not ko:
        return False
    # 너무 긴 문자열은 카드 제목이 아니라 주변 텍스트가 섞인 경우일 수 있음
    if len(text) > 180:
        return False
    return True


def split_values_from_lines(lines: List[str]) -> List[str]:
    raw = " ".join(normalize_text(line) for line in lines if normalize_text(line))
    raw = raw.replace("，", ",")
    raw = raw.replace("ㆍ", ",").replace("·", ",")
    raw = re.sub(r"\s*,\s*", ",", raw)
    raw = raw.strip(" ,，、")

    if not raw:
        return []

    banned = {
        "증상", "관련질환", "진료과", "질환설명", "의료진", "뉴스룸",
        "로그인", "회원가입", "나의차트", "병원둘러보기", "오시는길", "Language",
    }

    parts = [p.strip(" ,，、") for p in raw.split(",")]
    cleaned = []
    for part in parts:
        part = normalize_text(part)
        if not part or part in banned:
            continue
        cleaned.append(part)
    return dedupe_keep_order(cleaned)


def label_starts(line: str, label: str) -> bool:
    line = normalize_text(line)
    return line == label or line.startswith(label + " ")


def label_remainder(line: str, label: str) -> str:
    line = normalize_text(line)
    if line == label:
        return ""
    if line.startswith(label + " "):
        return normalize_text(line[len(label):])
    return ""


def find_label_index(lines: List[str], label: str, start: int = 0, max_gap: Optional[int] = None) -> Optional[int]:
    end = len(lines)
    if max_gap is not None:
        end = min(len(lines), start + max_gap + 1)
    for idx in range(start, end):
        if label_starts(lines[idx], label):
            return idx
    return None


def find_exact_index(lines: List[str], label: str, start: int = 0) -> Optional[int]:
    for idx in range(start, len(lines)):
        if normalize_text(lines[idx]) == label:
            return idx
    return None


def find_any_index(lines: List[str], targets: Iterable[str], start: int = 0) -> Optional[int]:
    targets = list(targets)
    for idx in range(start, len(lines)):
        line = normalize_text(lines[idx])
        for target in targets:
            if line == target or label_starts(line, target):
                return idx
    return None


def get_lines_between_label(lines: List[str], start_idx: int, end_idx: int, start_label: str) -> List[str]:
    values = []
    rem = label_remainder(lines[start_idx], start_label)
    if rem:
        values.append(rem)
    values.extend(lines[start_idx + 1:end_idx])
    return values

# ------------------------------------------------------------
# 3. HTTP / HTML
# ------------------------------------------------------------
last_request_time = 0.0


def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": (
                "Mozilla/5.0 "
                "(compatible; MunjinToktokMVP/1.0; educational one-time crawler)"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.7,en;q=0.6",
        }
    )
    retry = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=0.8,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session


def polite_wait() -> None:
    global last_request_time
    elapsed = time.time() - last_request_time
    wait = REQUEST_DELAY_SEC - elapsed
    if wait > 0:
        time.sleep(wait)
    last_request_time = time.time()


def fetch_html(session: requests.Session, url: str, timeout: float = 20.0) -> str:
    cpath = cache_path_for_url(url)
    if USE_CACHE and (not FORCE_REFRESH) and cpath.exists():
        return cpath.read_text(encoding="utf-8")

    polite_wait()
    resp = session.get(url, timeout=timeout)
    resp.raise_for_status()
    if not resp.encoding or resp.encoding.lower() in {"iso-8859-1", "ascii"}:
        resp.encoding = resp.apparent_encoding or "utf-8"
    html = resp.text
    if USE_CACHE:
        cpath.write_text(html, encoding="utf-8")
    return html


def soup_from_html(html: str) -> BeautifulSoup:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return soup


def extract_lines(node) -> List[str]:
    text = node.get_text("\n", strip=True)
    lines = []
    for raw_line in text.split("\n"):
        line = normalize_text(raw_line)
        if line:
            lines.append(line)
    return lines

# ------------------------------------------------------------
# 4. 목록 페이지 파싱
# ------------------------------------------------------------
def parse_total_count(lines: List[str]) -> Optional[int]:
    joined = "\n".join(lines)
    match = re.search(r"검색결과가\s*총\s*(\d+)건", joined)
    return int(match.group(1)) if match else None


def parse_summary_fields_from_lines(lines: List[str]) -> Dict[str, List[str]]:
    i = find_label_index(lines, "증상", start=0)
    if i is None:
        return {"symptoms": [], "related_diseases": [], "departments": []}
    j = find_label_index(lines, "관련질환", start=i + 1)
    if j is None:
        return {"symptoms": [], "related_diseases": [], "departments": []}
    k = find_label_index(lines, "진료과", start=j + 1)
    if k is None:
        return {"symptoms": [], "related_diseases": [], "departments": []}

    stop = find_any_index(
        lines,
        ["질환설명", "의료진", "뉴스룸", "정의", "로그인", "회원가입"],
        start=k + 1,
    )
    if stop is None:
        stop = len(lines)

    symptom_lines = get_lines_between_label(lines, i, j, "증상")
    related_lines = get_lines_between_label(lines, j, k, "관련질환")
    department_lines = get_lines_between_label(lines, k, stop, "진료과")

    return {
        "symptoms": split_values_from_lines(symptom_lines),
        "related_diseases": split_values_from_lines(related_lines),
        "departments": split_values_from_lines(department_lines),
    }


def find_best_card_container(anchor):
    """
    제목 anchor 주변에서 카드 컨테이너를 찾는다.
    li를 우선하고, 실패하면 부모를 타고 올라가며 증상/관련질환/진료과가 있는 작은 영역을 채택한다.
    """
    # li가 있으면 가장 안정적
    li = anchor.find_parent("li")
    if li is not None:
        text = normalize_text(li.get_text(" ", strip=True))
        if all(label in text for label in ["증상", "관련질환", "진료과"]):
            return li

    node = anchor
    for _ in range(12):
        node = node.parent
        if node is None:
            break
        text = normalize_text(node.get_text(" ", strip=True))
        if not all(label in text for label in ["증상", "관련질환", "진료과"]):
            continue
        title_like_count = 0
        for a in node.find_all("a", href=True):
            href = a.get("href", "")
            title = normalize_text(a.get_text(" ", strip=True))
            if "diseaseDetail.do" in href and is_disease_title_text(title):
                title_like_count += 1
        if title_like_count <= 1 and len(text) < 5000:
            return node
    return None


def extract_title_anchors(soup: BeautifulSoup) -> List[Dict[str, Any]]:
    result = []
    seen_content_ids = set()

    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        text = normalize_text(a.get_text(" ", strip=True))
        if "diseaseDetail.do" not in href:
            continue
        if not is_disease_title_text(text):
            continue
        full_url = urljoin(BASE_URL, href)
        content_id = parse_content_id(full_url)
        if not content_id:
            continue
        if content_id in seen_content_ids:
            continue
        seen_content_ids.add(content_id)
        result.append({"content_id": content_id, "url": full_url, "title": text, "anchor": a})

    return result


def fallback_parse_fields_by_global_lines(cards: List[Dict[str, Any]], lines: List[str]) -> List[Dict[str, Any]]:
    title_positions = {}
    for idx, line in enumerate(lines):
        for card in cards:
            if line == card["display_name"]:
                title_positions[card["content_id"]] = idx

    sorted_cards = sorted(cards, key=lambda c: title_positions.get(c["content_id"], 10**9))

    for pos, card in enumerate(sorted_cards):
        if card["symptoms"] and card["related_diseases"] and card["departments"]:
            continue
        start = title_positions.get(card["content_id"])
        if start is None:
            continue
        if pos + 1 < len(sorted_cards):
            next_start = title_positions.get(sorted_cards[pos + 1]["content_id"], len(lines))
        else:
            next_start = len(lines)
        segment = lines[start:next_start]
        fields = parse_summary_fields_from_lines(segment)
        if not card["symptoms"]:
            card["symptoms"] = fields["symptoms"]
        if not card["related_diseases"]:
            card["related_diseases"] = fields["related_diseases"]
        if not card["departments"]:
            card["departments"] = fields["departments"]
    return cards


def parse_list_page(html: str) -> Tuple[List[Dict[str, Any]], Optional[int]]:
    soup = soup_from_html(html)
    lines = extract_lines(soup)
    total_count = parse_total_count(lines)

    title_anchors = extract_title_anchors(soup)
    cards = []

    for item in title_anchors:
        title = item["title"]
        name_ko, name_en = split_disease_name(title)
        card = {
            "content_id": item["content_id"],
            "disease_id": make_disease_id(item["content_id"]),
            "source_url": item["url"],
            "display_name": title,
            "name_ko": name_ko,
            "name_en": name_en,
            "symptoms": [],
            "related_diseases": [],
            "departments": [],
        }

        container = find_best_card_container(item["anchor"])
        if container is not None:
            card_lines = extract_lines(container)
            fields = parse_summary_fields_from_lines(card_lines)
            card["symptoms"] = fields["symptoms"]
            card["related_diseases"] = fields["related_diseases"]
            card["departments"] = fields["departments"]

        cards.append(card)

    cards = fallback_parse_fields_by_global_lines(cards, lines)
    return cards, total_count


def collect_list_cards(session: requests.Session) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    all_cards = []
    seen_content_ids = set()
    total_count = None
    page_reports = []
    empty_page_count = 0

    for page_index in range(1, MAX_LIST_PAGES + 1):
        page_url = set_query_params(LIST_URL, diseaseKindId=DISEASE_KIND_ID, pageIndex=page_index)
        html = fetch_html(session, page_url)
        cards, page_total = parse_list_page(html)

        if total_count is None and page_total is not None:
            total_count = page_total

        new_count = 0
        for card in cards:
            cid = card["content_id"]
            if cid in seen_content_ids:
                continue
            seen_content_ids.add(cid)
            all_cards.append(card)
            new_count += 1

        page_reports.append({
            "page_index": page_index,
            "url": page_url,
            "cards_on_page": len(cards),
            "new_cards": new_count,
            "cumulative_cards": len(all_cards),
        })
        print(f"[LIST] page={page_index}, cards={len(cards)}, new={new_count}, total={len(all_cards)}")

        if new_count == 0:
            empty_page_count += 1
        else:
            empty_page_count = 0

        if total_count is not None and len(all_cards) >= total_count:
            break
        if empty_page_count >= 2:
            break

    report = {
        "list_url": LIST_URL,
        "kind_id": DISEASE_KIND_ID,
        "expected_total_count": total_count,
        "collected_count": len(all_cards),
        "page_reports": page_reports,
    }
    return all_cards, report

# ------------------------------------------------------------
# 5. 상세 페이지 sections 파싱
# ------------------------------------------------------------
SECTION_LABEL_TO_KEY = {
    "정의": "definition",
    "원인": "cause",
    "증상": "symptom",
    "진단": "diagnosis",
    "치료": "treatment",
    "경과": "course",
    "주의사항": "caution",
}
SECTION_LABELS = list(SECTION_LABEL_TO_KEY.keys())
SECTION_TERMINATORS = [
    "콘텐츠 제공 문의하기",
    "서울아산병원은",
    "다른질환보기",
    "공유하기",
    "현재 페이지를 공유하기",
    "현재 페이지를 인쇄하기",
    "로그인",
    "회원가입",
]


def extract_detail_title(soup: BeautifulSoup, lines: List[str], fallback: str = "") -> str:
    title_tag = soup.find("title")
    if title_tag:
        title = normalize_text(title_tag.get_text(" "))
        title = normalize_text(title.split("|")[0])
        if title and title != "질환백과":
            return title
    for line in lines:
        if is_disease_title_text(line):
            return line
    return normalize_text(fallback)


def extract_detail_sections(lines: List[str]) -> Dict[str, str]:
    sections = {key: "" for key in SECTION_LABEL_TO_KEY.values()}

    definition_idx = find_exact_index(lines, "정의", start=0)
    if definition_idx is None:
        return sections

    positions = []
    for label in SECTION_LABELS:
        idx = find_exact_index(lines, label, start=definition_idx)
        if idx is not None:
            positions.append((label, idx))
    positions.sort(key=lambda x: x[1])

    for pos, (label, idx) in enumerate(positions):
        next_idx = positions[pos + 1][1] if pos + 1 < len(positions) else len(lines)
        terminator_idx = find_any_index(lines, SECTION_TERMINATORS, start=idx + 1)
        end_idx = min(next_idx, terminator_idx) if terminator_idx is not None else next_idx

        content_lines = []
        for line in lines[idx + 1:end_idx]:
            line = normalize_text(line)
            if not line:
                continue
            if line in SECTION_LABELS or line in SECTION_TERMINATORS:
                continue
            if re.fullmatch(r"\d{1,2}:\d{2}", line):
                continue
            content_lines.append(line)

        # 정보 손실 방지를 위해 자르지 않고 줄바꿈 보존
        sections[SECTION_LABEL_TO_KEY[label]] = "\n".join(content_lines).strip()

    return sections


def parse_detail_page(html: str, url: str, fallback_title: str = "") -> Dict[str, Any]:
    soup = soup_from_html(html)
    lines = extract_lines(soup)
    parsed_title = extract_detail_title(soup, lines, fallback=fallback_title)
    parsed_name_ko, parsed_name_en = split_disease_name(parsed_title)
    detail_summary = parse_summary_fields_from_lines(lines)
    sections = extract_detail_sections(lines)
    return {
        "parsed_title": parsed_title,
        "parsed_name_ko": parsed_name_ko,
        "parsed_name_en": parsed_name_en,
        "summary": detail_summary,
        "sections": sections,
    }

# ------------------------------------------------------------
# 6. 실행 함수
# ------------------------------------------------------------
def run_crawl() -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    session = make_session()
    started_at = now_utc_iso()

    list_cards, list_report = collect_list_cards(session)

    print("\n목록 수집 완료")
    print("목록 표시 총 건수:", list_report.get("expected_total_count"))
    print("수집된 질환 카드 수:", len(list_cards))

    # 문제 되었던 nested parentheses 제목 확인
    print("\n[CHECK] 괄호 중첩 질환 제목 확인")
    for keyword in ["병원 감염성 폐렴", "신종 플루", "객혈"]:
        found = [c for c in list_cards if keyword in c.get("display_name", "") or keyword == c.get("name_ko")]
        print(f"- {keyword}: {len(found)}건")
        for item in found:
            print({
                "content_id": item["content_id"],
                "display_name": item["display_name"],
                "name_ko": item["name_ko"],
                "name_en": item["name_en"],
                "symptoms": item["symptoms"],
                "departments": item["departments"],
            })

    if DETAIL_LIMIT is not None:
        cards_to_crawl = list_cards[:DETAIL_LIMIT]
        print(f"\nDETAIL_LIMIT={DETAIL_LIMIT} 적용: 상세 {len(cards_to_crawl)}개만 크롤링")
    else:
        cards_to_crawl = list_cards
        print(f"\n전체 상세 크롤링 시작: {len(cards_to_crawl)}개")

    diseases = []
    errors = []

    for idx, card in enumerate(cards_to_crawl, start=1):
        try:
            print(f"[DETAIL] {idx}/{len(cards_to_crawl)} {card['display_name']} / contentId={card['content_id']}")
            html = fetch_html(session, card["source_url"])
            detail = parse_detail_page(html=html, url=card["source_url"], fallback_title=card["display_name"])

            # 목록 필드를 우선 사용. 목록 필드가 비어 있으면 상세 상단 summary fallback.
            disease = {
                "disease_id": card["disease_id"],
                "content_id": card["content_id"],
                "source_url": card["source_url"],
                "category": DISEASE_KIND_NAME,
                "disease_kind_id": DISEASE_KIND_ID,
                "name_ko": card["name_ko"] or detail["parsed_name_ko"],
                "name_en": card["name_en"] or detail["parsed_name_en"],
                "display_name": card["display_name"] or detail["parsed_title"],
                "symptoms": card["symptoms"] or detail["summary"]["symptoms"],
                "related_diseases": card["related_diseases"] or detail["summary"]["related_diseases"],
                "departments": card["departments"] or detail["summary"]["departments"],
                "sections": detail["sections"],
            }
            diseases.append(disease)
        except Exception as exc:
            error = {
                "content_id": card.get("content_id", ""),
                "source_url": card.get("source_url", ""),
                "display_name": card.get("display_name", ""),
                "error": f"{type(exc).__name__}: {exc}",
            }
            errors.append(error)
            print("[ERROR]", error)

    # content_id 기준 중복 제거
    unique = {}
    for disease in diseases:
        key = disease.get("content_id") or disease.get("source_url")
        unique[key] = disease
    diseases = list(unique.values())

    finished_at = now_utc_iso()
    expected_total = list_report.get("expected_total_count")

    crawl_report = {
        "crawler": "amc_diseases_crawler_fixed_nested_parentheses",
        "started_at": started_at,
        "finished_at": finished_at,
        "source": "서울아산병원 질환백과",
        "list_url": LIST_URL,
        "kind_id": DISEASE_KIND_ID,
        "kind_name": DISEASE_KIND_NAME,
        "request_delay_sec": REQUEST_DELAY_SEC,
        "use_cache": USE_CACHE,
        "force_refresh": FORCE_REFRESH,
        "detail_limit": DETAIL_LIMIT,
        "list_report": list_report,
        "disease_count": len(diseases),
        "expected_total_count": expected_total,
        "count_matches_expected": (expected_total is None or len(list_cards) == expected_total),
        "errors": errors,
    }

    write_json(OUT_DIR / "diseases.json", diseases)
    write_json(OUT_DIR / "crawl_report.json", crawl_report)

    # 확인용 CSV도 함께 저장
    rows = []
    for d in diseases:
        rows.append({
            "disease_id": d.get("disease_id"),
            "content_id": d.get("content_id"),
            "name_ko": d.get("name_ko"),
            "name_en": d.get("name_en"),
            "symptoms": ", ".join(d.get("symptoms", [])),
            "related_disease_count": len(d.get("related_diseases", [])),
            "departments": ", ".join(d.get("departments", [])),
            "section_keys_filled": ", ".join([k for k, v in d.get("sections", {}).items() if normalize_text(v)]),
            "symptom_count": len(d.get("symptoms", [])),
        })
    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR / "diseases_debug_table.csv", index=False, encoding="utf-8-sig")

    print("\n저장 완료")
    print("diseases.json:", OUT_DIR / "diseases.json")
    print("crawl_report.json:", OUT_DIR / "crawl_report.json")
    print("diseases_debug_table.csv:", OUT_DIR / "diseases_debug_table.csv")
    print("질환 수:", len(diseases))
    print("목록 수집 카드 수:", len(list_cards))
    print("목록 표시 총 건수:", expected_total)
    print("에러 수:", len(errors))

    if expected_total is not None and len(list_cards) != expected_total:
        print("\n[WARN] 목록 표시 총 건수와 수집 카드 수가 다릅니다. 파서 또는 사이트 구조를 다시 확인하세요.")
    else:
        print("\n[OK] 목록 표시 총 건수와 수집 카드 수가 일치합니다.")

    return diseases, crawl_report


# ------------------------------------------------------------
# 7. 직접 실행 시 크롤링 수행
# ------------------------------------------------------------
if __name__ == "__main__":
    diseases, crawl_report = run_crawl()

    # 콘솔 확인용
    df_rows = []
    for d in diseases:
        df_rows.append({
            "content_id": d.get("content_id"),
            "name_ko": d.get("name_ko"),
            "name_en": d.get("name_en"),
            "symptoms": ", ".join(d.get("symptoms", [])),
            "departments": ", ".join(d.get("departments", [])),
            "symptom_count": len(d.get("symptoms", [])),
        })
    df = pd.DataFrame(df_rows)
    print(df.head(20).to_string(index=False))

[OK] requests
[OK] beautifulsoup4
[OK] pandas
설정 완료
OUT_DIR: C:\Users\CGB\AIHub\output_amc_diseases_fixed
CACHE_DIR: C:\Users\CGB\AIHub\output_amc_diseases_fixed\cache_html
[LIST] page=1, cards=20, new=20, total=20
[LIST] page=2, cards=20, new=20, total=40
[LIST] page=3, cards=20, new=20, total=60
[LIST] page=4, cards=3, new=3, total=63

목록 수집 완료
목록 표시 총 건수: 63
수집된 질환 카드 수: 63

[CHECK] 괄호 중첩 질환 제목 확인
- 병원 감염성 폐렴: 1건
{'content_id': '31743', 'display_name': '병원 감염성 폐렴(Hospital acquired pneumonia (HAP))', 'name_ko': '병원 감염성 폐렴', 'name_en': 'Hospital acquired pneumonia (HAP)', 'symptoms': ['흉통', '근육통', '가래', '열', '오한', '기침'], 'departments': ['호흡기내과']}
- 신종 플루: 1건
{'content_id': '31665', 'display_name': '신종 플루(Novel swine-origin influenza A(H1N1))', 'name_ko': '신종 플루', 'name_en': 'Novel swine-origin influenza A(H1N1)', 'symptoms': ['오한', '두통', '목의 통증', '구토', '근육통', '설사'], 'departments': ['감염내과', '호흡기내과']}
- 객혈: 1건
{'content_id': '31850', 'display_name': '객혈(Hemoptysis)', 'name_ko': '객혈', 'n